# IPR of the memory: plots

Loads the `IPR(T)` heatmap written by `wehrl_entropy_poster.ipynb`
(`results/IPR_array_poster_<date>.json`) and plots it. Nothing is simulated
here, so this needs neither the cluster nor jax.

`IPR = sum_k <k|rho_a(T)|k>^2` is the inverse participation ratio of the
memory's reduced state in the Fock basis, at `T = 0.2 / kappa`
(`n_a = 29`, `n_b = 17`, poster parameters, interaction frame).

The JSON holds only the values: a list of rows, one per `epsilon_0`, one entry
per `epsilon_d / g2`. The two axes are rebuilt below from the same definitions
as in `wehrl_entropy_poster.ipynb`. If that grid changes there, change it here
too. The shape check catches a mismatch in size, but not in values.

**`g2` changed on 2026-10-12.** The simulating notebooks now use `g2 =
sin(epsilon_0) E_J phi_a^2 phi_b` (static pump), in place of `J_1(epsilon_0) E_J
phi_a^2 phi_b`. Result files written before that change were computed with
`J_1`: in them, `epsilon_d / g2` is `epsilon_d / g2_J1`, and the true ratio is
`J_1 / sin` (about 1/2) times that, so their cats are about half the size the
axis says. A re-run on `12_10_2026` overwrites the files of that date, so check
the file's modification time when in doubt.

In [ ]:
import json
import pathlib
import re
import sys
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/Code_python")
import Plot_setting
from Plot_setting import Perso_colormap
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# The results/ folder next to this notebook (or under the workspace root, if that
# is the kernel's working directory).
here = pathlib.Path.cwd()
results_dir = Path("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/data/otoc")

# None: the most recent IPR_array_poster_<dd_mm_yyyy>.json. Or a date, e.g. "12_10_2026".
DATE = None

def _date_key(path):
    d, m, y = re.search(r"(\d{2})_(\d{2})_(\d{4})\.json$", path.name).groups()
    return (int(y), int(m), int(d))

if DATE is None:
    candidates = sorted(results_dir.glob("IPR_array_poster_*.json"), key=_date_key)
    if not candidates:
        raise FileNotFoundError(f"no IPR_array_poster_*.json in {results_dir}; "
                                "run wehrl_entropy_poster.ipynb first")
    ipr_path = candidates[-1]
else:
    ipr_path = results_dir / f"IPR_array_poster_{DATE}.json"

IPR_arr = np.array(json.load(open(ipr_path)), dtype=float)
print(f"loaded {ipr_path.name}: shape {IPR_arr.shape}, "
      f"{np.isnan(IPR_arr).sum()} failed point(s), "
      f"IPR in [{np.nanmin(IPR_arr):.4f}, {np.nanmax(IPR_arr):.4f}]")

In [ ]:
# The heatmap grid of wehrl_entropy_poster.ipynb (= OTOCS_poster.ipynb).
epsilon_0_list = list(np.concatenate((np.arange(0.2, .6, 0.07), np.arange(0.6, 1.1, 0.03))))
epsilon_0_arr = np.array(epsilon_0_list, dtype=np.float32)
ratio_list = np.arange(2, 10, 1)
n_a, n_b = 29, 17

assert IPR_arr.shape == (len(epsilon_0_arr), len(ratio_list)), (
    f"{ipr_path.name} is {IPR_arr.shape}, the grid here is "
    f"{(len(epsilon_0_arr), len(ratio_list))}: the two notebooks disagree on the grid")

## `IPR(T)` over the pump amplitude and the drive strength

`epsilon_0` is not evenly spaced (steps of `0.07` up to `0.6`, then `0.03`),
so the cells are drawn at their true positions with `pcolormesh`, not
`imshow`. The heatmaps in the other notebooks use `imshow`, which spreads the
23 columns evenly.

In [ ]:
def _edges(x):
    """Cell edges around the centers x: the midpoints, extended at both ends."""
    x = np.asarray(x, dtype=float)
    mid = (x[1:] + x[:-1]) / 2
    return np.concatenate(([x[0] - (mid[0] - x[0])], mid, [x[-1] + (x[-1] - mid[-1])]))

heatmap_date = "12_10_2026"

fig, ax = plt.subplots(figsize=(6.4, 4.6), constrained_layout=True)
im = ax.pcolormesh(_edges(epsilon_0_arr), _edges(ratio_list), IPR_arr.T, cmap=Perso_colormap)
ax.set_xlabel(r'$\varepsilon_p$' + ' (rad)')
ax.set_ylabel(r'$\varepsilon_d / g_2$')
ax.set_title(r'IPR$(T)$, $T=2/\kappa_b$, $n_a=$' + str(n_a) + ', ' + r'$n_b=$' + str(n_b))
fig.colorbar(im, ax=ax, label=r'IPR $=\sum_k \langle k|\rho_a(T)|k\rangle^2$')
plt.savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/article_chaos/figures_article/"
            f"IPR_heatmap_{heatmap_date}.pdf")
plt.savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/otoc/"
            f"IPR_heatmap_{heatmap_date}.pdf")
plt.show()

## Cuts at fixed drive strength

The same data, one curve per `epsilon_d / g2`, coloured from light (`2`) to
dark (`9`).

In [ ]:
colors = Perso_colormap(np.linspace(0.9, 0.1, len(ratio_list)))

fig, ax = plt.subplots(figsize=(6.4, 4.6), constrained_layout=True)
for j, (ratio, c) in enumerate(zip(ratio_list, colors)):
    ax.plot(epsilon_0_arr, IPR_arr[:, j], "o-", color=c, lw=2, ms=4,
            label=rf'$\varepsilon_d/g_2 = {ratio:g}$')
ax.set_xlabel(r'$\varepsilon_p$' + ' (rad)')
ax.set_ylabel(r'IPR $=\sum_k \langle k|\rho_a(T)|k\rangle^2$')
ax.set_title(r'IPR$(T)$, $T=0.2/\kappa_b$, $n_a=$' + str(n_a) + ', ' + r'$n_b=$' + str(n_b))
ax.grid(alpha=0.3)
ax.legend(fontsize=8, ncol=2)
plt.show()